In [1]:
import requests
import pandas as pd

def fetch_world_bank_data(indicator: str, country: str, start: int, end: int) -> pd.DataFrame:
    """
    Fetch data from World Bank API.
    indicator: e.g. 'SP.POP.TOTL' = total population
    country:   e.g. 'RW' = Rwanda
    """
    url = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"
    params = {
        "date": f"{start}:{end}",
        "format": "json",
        "per_page": 100
    }

    response = requests.get(url, params=params)
    response.raise_for_status()   # raises exception if status != 200

    raw = response.json()

    # World Bank returns [metadata, data] — we want index 1
    records = raw[1]

    df = pd.DataFrame([{
        "year": r["date"],
        "value": r["value"],
        "country": r["country"]["value"]
    } for r in records if r["value"] is not None])

    return df

# Fetch Rwanda population 2000-2020
df = fetch_world_bank_data("SP.POP.TOTL", "RW", 2000, 2020)
print(df.head())

   year     value country
0  2020  13065837  Rwanda
1  2019  12776103  Rwanda
2  2018  12487996  Rwanda
3  2017  12202060  Rwanda
4  2016  11919183  Rwanda


In [2]:
import requests
import logging
import time

def fetch_with_retry(url: str, params: dict, max_retries: int = 3) -> dict:
    """Fetch from API with retry logic on failure."""
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            logging.warning(f"Attempt {attempt}: Request timed out")
        except requests.exceptions.HTTPError as e:
            logging.error(f"HTTP error: {e}")
            break   # don't retry on HTTP errors (404, 403 etc.)
        except requests.exceptions.ConnectionError:
            logging.warning(f"Attempt {attempt}: Connection failed")

        if attempt < max_retries:
            time.sleep(2 ** attempt)   # wait 2s, 4s, 8s — exponential backoff

    logging.error("All retries failed")
    return {}